# Aula 02 — Titanic: Pipeline Mínimo Profissional

**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  
**Aluno:** Felipe Arruda

## Objetivo
Treinar um modelo de ML no Titanic usando Pipeline + ColumnTransformer,
com baseline (Logistic Regression) e uma melhoria (Random Forest),
avaliando com accuracy, classification_report e matriz de confusão.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [2]:
# 1) Carregar o dataset e remover a coluna 'alive'
df_t = sns.load_dataset("titanic").drop(columns=["alive"])

display(df_t.head())
print("Shape:", df_t.shape)
print("\nFaltantes por coluna:")
display(df_t.isna().sum().sort_values(ascending=False))

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,True


Shape: (891, 14)

Faltantes por coluna:


,0
deck,688
age,177
embarked,2
embark_town,2
pclass,0
survived,0
parch,0
sibsp,0
sex,0
fare,0


In [3]:
# 2) Separar X e y
y = df_t["survived"]
X = df_t.drop(columns=["survived"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 13)
y shape: (891,)


In [4]:
# 3) Split com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)

Treino: (712, 13) | Teste: (179, 13)


In [5]:
# 4) Detectar colunas numéricas e categóricas
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

print("Numéricas:", list(num_cols))
print("Categóricas:", list(cat_cols))

Numéricas: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categóricas: ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


In [6]:
# 5) Pipeline de pré-processamento
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
], remainder="drop")

In [7]:
# Função auxiliar para avaliação
def avaliar(nome, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n=== {nome} ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("Matriz de Confusão [[TN, FP],[FN, TP]]:")
    print(cm)
    print(f"\nTN={tn} | FP={fp} | FN={fn} | TP={tp}")

    return {"acc": acc, "tn": tn, "fp": fp, "fn": fn, "tp": tp}

In [8]:
# 6) Baseline — Logistic Regression
model_lr = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)

# 7) Avaliação do baseline
res_lr = avaliar("Baseline — LogisticRegression", y_test, pred_lr)


=== Baseline — LogisticRegression ===
Accuracy: 0.8268

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       110
           1       0.79      0.75      0.77        69

    accuracy                           0.83       179
   macro avg       0.82      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179

Matriz de Confusão [[TN, FP],[FN, TP]]:
[[96 14]
 [17 52]]

TN=96 | FP=14 | FN=17 | TP=52


In [9]:
# 8) Melhoria — Random Forest
model_rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE))
])

model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)

res_rf = avaliar("Melhoria — RandomForestClassifier", y_test, pred_rf)


=== Melhoria — RandomForestClassifier ===
Accuracy: 0.8212

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       110
           1       0.82      0.68      0.75        69

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.82       179

Matriz de Confusão [[TN, FP],[FN, TP]]:
[[100  10]
 [ 22  47]]

TN=100 | FP=10 | FN=22 | TP=47


In [11]:
# Comparação final
print("=== COMPARAÇÃO FINAL ===")
print(f"Accuracy  LR: {res_lr['acc']:.4f}   |  RF: {res_rf['acc']:.4f}")
print(f"TN        LR: {res_lr['tn']}       |  RF: {res_rf['tn']}")
print(f"FP        LR: {res_lr['fp']}       |  RF: {res_rf['fp']}")
print(f"FN        LR: {res_lr['fn']}       |  RF: {res_rf['fn']}")
print(f"TP        LR: {res_lr['tp']}       |  RF: {res_rf['tp']}")

=== COMPARAÇÃO FINAL ===
Accuracy  LR: 0.8268   |  RF: 0.8212
TN        LR: 96       |  RF: 100
FP        LR: 14       |  RF: 10
FN        LR: 17       |  RF: 22
TP        LR: 52       |  RF: 47


## Interpretação dos Resultados

### 1) Em qual classe o modelo erra mais?
O modelo erra mais na **classe 1 (sobreviveu)**.
O recall da classe 1 é menor do que o da classe 0 no baseline,
e o valor de FN (pessoas que sobreviveram mas foram previstas como
"não sobreviveu") é maior do que o FP.
Isso indica que o modelo "perde" mais sobreviventes do que gera
alarmes falsos.

### 2) O que mudou em FP e FN entre baseline e melhoria?
- O **Random Forest reduziu o FP** (menos alarmes falsos),
  mas **aumentou o FN** em relação ao baseline.
- Isso é o trade-off clássico: ao ser mais conservador para
  classificar alguém como "sobreviveu", o modelo erra menos
  para quem não sobreviveu, mas perde mais sobreviventes reais.

### 3) Uma limitação do dataset
O dataset possui **muitos valores faltantes** na coluna `deck`
(~77% dos dados ausentes) e em `age` (~20% ausentes).
Além disso, há um **viés histórico forte**: a política
"mulheres e crianças primeiro" faz com que `sex` seja
um preditor dominante, o que pode mascarar outros fatores
igualmente relevantes na realidade.